# Sign Language Model Training

Trains the `SignLanguageModel` using CrossEntropyLoss with Two-Stage Transfer Learning.

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
import os
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

import sys
sys.path.append('.')
from modules.models import SignLanguageModel

Using device: cuda


In [2]:
DATA_DIR = "data/processed"
with open(os.path.join(DATA_DIR, "core_label_map.json"), "r", encoding="utf-8") as f:
    num_classes = len(json.load(f))
print(f"Total Core Classes: {num_classes}")

X_train_raw = np.load(os.path.join(DATA_DIR, "X_train.npy"))
y_train_raw = np.load(os.path.join(DATA_DIR, "y_train.npy"))

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw, test_size=0.1, random_state=42, stratify=y_train_raw
)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

Total Core Classes: 112
Train: (5040, 30, 150), Val: (560, 30, 150)


In [3]:
class KSLDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(KSLDataset(X_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(KSLDataset(X_val, y_val), batch_size=64, shuffle=False)

In [4]:
model = SignLanguageModel(num_classes=num_classes, hidden_dim=64).to(device)
criterion = nn.CrossEntropyLoss()

def train_epoch(optim):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optim.zero_grad()
        
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optim.step()
        
        total_loss += loss.item() * X_batch.size(0)
        _, pred = torch.max(logits, 1)
        correct += (pred == y_batch).sum().item()
        total += X_batch.size(0)
    return total_loss / total, correct / total

def eval_epoch():
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            
            total_loss += loss.item() * X_batch.size(0)
            _, pred = torch.max(logits, 1)
            correct += (pred == y_batch).sum().item()
            total += X_batch.size(0)
    return total_loss / total, correct / total

In [5]:
print("[Stage 1] Training GRU & Classifier only (Freezing CNN)...")
for param in model.encoder.conv_block.parameters():
    param.requires_grad = False

optimizer_s1 = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

best_acc = 0.0
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
for epoch in range(1, 11):
    t_loss, t_acc = train_epoch(optimizer_s1)
    v_loss, v_acc = eval_epoch()
    print(f"Epoch {epoch:02d} | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")
    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)
    if v_acc > best_acc:
        best_acc = v_acc
        os.makedirs("models", exist_ok=True)
        torch.save(model.state_dict(), "models/sign_language_gru_best.pth")

[Stage 1] Training GRU & Classifier only (Freezing CNN)...
Epoch 01 | Train Loss: 4.6850 Acc: 0.0161 | Val Loss: 4.4692 Acc: 0.0286
Epoch 02 | Train Loss: 4.1498 Acc: 0.0462 | Val Loss: 3.7620 Acc: 0.0696
Epoch 03 | Train Loss: 3.7414 Acc: 0.0637 | Val Loss: 3.4879 Acc: 0.1357
Epoch 04 | Train Loss: 3.4705 Acc: 0.1099 | Val Loss: 3.1632 Acc: 0.2161
Epoch 05 | Train Loss: 3.1417 Acc: 0.1855 | Val Loss: 2.8197 Acc: 0.2893
Epoch 06 | Train Loss: 2.8391 Acc: 0.2464 | Val Loss: 2.4927 Acc: 0.4179
Epoch 07 | Train Loss: 2.5332 Acc: 0.3183 | Val Loss: 2.1946 Acc: 0.5054
Epoch 08 | Train Loss: 2.2821 Acc: 0.3825 | Val Loss: 1.9513 Acc: 0.5518
Epoch 09 | Train Loss: 2.0653 Acc: 0.4395 | Val Loss: 1.7469 Acc: 0.6036
Epoch 10 | Train Loss: 1.8831 Acc: 0.4895 | Val Loss: 1.5743 Acc: 0.6607


In [6]:
print("\n[Stage 2] Fine-tuning Entire Network (Unfreezing CNN)...")
model.load_state_dict(torch.load("models/sign_language_gru_best.pth"))

for param in model.encoder.conv_block.parameters():
    param.requires_grad = True

optimizer_s2 = torch.optim.AdamW(model.parameters(), lr=0.0001) # Lower LR for fine-tuning

for epoch in range(11, 21):
    t_loss, t_acc = train_epoch(optimizer_s2)
    v_loss, v_acc = eval_epoch()
    print(f"Epoch {epoch:02d} | Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")
    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)
    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), "models/sign_language_gru_best.pth")

print(f"\nTraining Complete! Best Val Acc: {best_acc:.4f}")
# Rename to final target model
import os
os.makedirs("models", exist_ok=True)
with open("models/training_history.json", "w") as f:
    json.dump(history, f)
import shutil
shutil.copy("models/sign_language_gru_best.pth", "models/sign_language_gru.pth")


[Stage 2] Fine-tuning Entire Network (Unfreezing CNN)...
Epoch 11 | Train Loss: 1.7244 Acc: 0.5442 | Val Loss: 1.4715 Acc: 0.7054
Epoch 12 | Train Loss: 1.6559 Acc: 0.5685 | Val Loss: 1.4250 Acc: 0.7286
Epoch 13 | Train Loss: 1.6122 Acc: 0.5851 | Val Loss: 1.3782 Acc: 0.7339
Epoch 14 | Train Loss: 1.5881 Acc: 0.5911 | Val Loss: 1.3458 Acc: 0.7429
Epoch 15 | Train Loss: 1.5344 Acc: 0.6048 | Val Loss: 1.3137 Acc: 0.7571
Epoch 16 | Train Loss: 1.5018 Acc: 0.6325 | Val Loss: 1.2755 Acc: 0.7982
Epoch 17 | Train Loss: 1.4730 Acc: 0.6317 | Val Loss: 1.2338 Acc: 0.7714
Epoch 18 | Train Loss: 1.4363 Acc: 0.6466 | Val Loss: 1.2227 Acc: 0.7714
Epoch 19 | Train Loss: 1.4149 Acc: 0.6554 | Val Loss: 1.1773 Acc: 0.7946
Epoch 20 | Train Loss: 1.3859 Acc: 0.6762 | Val Loss: 1.1558 Acc: 0.8018

Training Complete! Best Val Acc: 0.8018


'models/sign_language_gru.pth'